<a href="https://colab.research.google.com/github/Nestor20193767/Grupo2---Proyecto/blob/main/ECG_TCNCVAE_Entrenamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🫀 TCN-CVAE – Entrenamiento y Generación Sintética de ECG
## Proyecto: Generación Sintética de Señales ECG · MIT-BIH Arrhythmia Database

---
**Prerequisito:** haber ejecutado `ECG_Avance2_Pipeline.ipynb` y tener los arrays en `/content/ecg_processed/`.

**Flujo de este notebook:**
1. Instalación y configuración GPU
2. Carga de los datos preprocesados (Avance 2)
3. `ECGDataset` + `DataLoader` PyTorch
4. Bloques constructores: `CausalDilatedBlock` + `TCNStack`
5. Módulos `TCNEncoder`, `Reparametrize`, `TCNDecoder`
6. Modelo raíz `TCNCVAE` + conteo de parámetros
7. Función de pérdida + β-scheduler
8. Bucle de entrenamiento con curvas de loss
9. Generación de latidos sintéticos condicionales
10. Validación: visualización, análisis espectral y t-SNE

## 1. Instalación y Configuración

In [1]:
# PyTorch ya viene preinstalado en Colab.
# Solo necesitamos verificar que la GPU está disponible.
!pip install -q torchinfo
print('✅ torchinfo instalado')

✅ torchinfo instalado


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
import os, time, copy, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import weight_norm
from torchinfo import summary

from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder

# ── Reproducibilidad ──────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Dispositivo ───────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Estilo matplotlib ─────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.35,
    'font.size': 11
})
print('✅ Imports completados')

🖥️  Dispositivo: cuda
   GPU: Tesla T4
   VRAM disponible: 15.6 GB
✅ Imports completados


## 2. Carga de Datos Preprocesados (desde Avance 2)

In [3]:
# ─────────────────────────────────────────────────────────
#  CARGA DE ARRAYS GUARDADOS EN EL AVANCE 2
# ─────────────────────────────────────────────────────────
DATA_DIR = '/content/ecg_processed'

# Si no existe el directorio, recordar al usuario que primero ejecute el Avance 2
if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(
        '❌ No se encontró /content/ecg_processed/.\n'
        '   Por favor ejecuta primero ECG_Avance2_Pipeline.ipynb para generar los arrays.'
    )

X_train = np.load(f'{DATA_DIR}/X_train.npy')   # shape: (N_train, 256)
y_train = np.load(f'{DATA_DIR}/y_train.npy')   # shape: (N_train,)  dtype: str
X_val   = np.load(f'{DATA_DIR}/X_val.npy')
y_val   = np.load(f'{DATA_DIR}/y_val.npy')
X_test  = np.load(f'{DATA_DIR}/X_test.npy')
y_test  = np.load(f'{DATA_DIR}/y_test.npy')

# Mapeo de clase string → índice entero (N=0, S=1, V=2, F=3, Q=4)
CLASS_NAMES = ['N', 'S', 'V', 'F', 'Q']
CLASS_IDX   = {c: i for i, c in enumerate(CLASS_NAMES)}
N_CLASSES   = len(CLASS_NAMES)

def labels_to_idx(y_str):
    return np.array([CLASS_IDX[c] for c in y_str], dtype=np.int64)

y_train_idx = labels_to_idx(y_train)
y_val_idx   = labels_to_idx(y_val)
y_test_idx  = labels_to_idx(y_test)

print('✅ Arrays cargados correctamente')
print(f'   X_train : {X_train.shape}  |  y_train: {y_train.shape}')
print(f'   X_val   : {X_val.shape}  |  y_val  : {y_val.shape}')
print(f'   X_test  : {X_test.shape}  |  y_test : {y_test.shape}')
print(f'\n   Distribución train: {dict(Counter(y_train))}')

FileNotFoundError: ❌ No se encontró /content/ecg_processed/.
   Por favor ejecuta primero ECG_Avance2_Pipeline.ipynb para generar los arrays.

## 3. Dataset y DataLoader PyTorch

In [ ]:
# ─────────────────────────────────────────────────────────
#  CLASE ECGDataset
#  Envuelve los arrays numpy en la interfaz Dataset de PyTorch.
#  Cada __getitem__ devuelve:
#    x      : tensor float32 shape (1, 256)  → señal con dim de canal
#    c      : tensor float32 shape (5,)      → one-hot de la clase AAMI
#    label  : tensor int64                   → índice numérico de clase
# ─────────────────────────────────────────────────────────
class ECGDataset(Dataset):
    def __init__(self, X, y_idx, n_classes=5):
        # X: numpy (N, 256)  y_idx: numpy (N,) int
        self.X        = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # (N,1,256)
        self.y        = torch.tensor(y_idx, dtype=torch.long)
        self.n_classes = n_classes

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x     = self.X[idx]          # (1, 256)
        label = self.y[idx]          # escalar int64
        # One-hot vector de condición
        c = F.one_hot(label, num_classes=self.n_classes).float()  # (5,)
        return x, c, label


# ── Hiperparámetros del DataLoader ────────────────────────
BATCH_SIZE  = 128
NUM_WORKERS = 2

train_ds = ECGDataset(X_train, y_train_idx)
val_ds   = ECGDataset(X_val,   y_val_idx)
test_ds  = ECGDataset(X_test,  y_test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))

print('✅ DataLoaders creados')
print(f'   Train  : {len(train_ds):,} muestras  →  {len(train_loader)} batches de {BATCH_SIZE}')
print(f'   Val    : {len(val_ds):,} muestras  →  {len(val_loader)} batches')
print(f'   Test   : {len(test_ds):,} muestras  →  {len(test_loader)} batches')

# Verificar shapes de un batch
x_b, c_b, lbl_b = next(iter(train_loader))
print(f'\n   Shapes de un batch de ejemplo:')
print(f'   x shape : {x_b.shape}   → (Batch, Canal=1, L=256)')
print(f'   c shape : {c_b.shape}  → (Batch, N_clases=5)')
print(f'   lbl shape: {lbl_b.shape} → (Batch,)')

## 4. Bloques Constructores: CausalDilatedBlock y TCNStack

In [ ]:
# ─────────────────────────────────────────────────────────
#  CausalDilatedBlock
#  Bloque residual TCN con:
#   - Convolución dilatada con padding SIMÉTRICO (no causal)
#     porque el ECG no es tiempo real; aprovechamos contexto
#     pasado Y futuro alrededor del pico R.
#   - WeightNorm para estabilizar el gradiente en capas profundas.
#   - Activación GELU (suaviza el gradiente vs ReLU).
#   - Skip connection con Conv1x1 si los canales cambian.
# ─────────────────────────────────────────────────────────
class CausalDilatedBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()
        # Padding simétrico = dilation * (kernel_size - 1) // 2
        pad = dilation * (kernel_size - 1) // 2

        self.net = nn.Sequential(
            weight_norm(nn.Conv1d(in_ch, out_ch, kernel_size,
                                  dilation=dilation, padding=pad)),
            nn.GELU(),
            nn.Dropout(dropout),
            weight_norm(nn.Conv1d(out_ch, out_ch, kernel_size,
                                  dilation=dilation, padding=pad)),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        # Skip connection: 1x1 conv si los canales difieren
        self.skip = (
            nn.Conv1d(in_ch, out_ch, kernel_size=1)
            if in_ch != out_ch else nn.Identity()
        )
        self.init_weights()

    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x) + self.skip(x)


# ─────────────────────────────────────────────────────────
#  TCNStack
#  Apila N bloques con dilataciones crecientes [1, 2, 4, 8].
#  El campo receptivo efectivo con kernel=3, 4 bloques:
#  RF = 1 + sum(2*(k-1)*d) = 1 + 2*2*(1+2+4+8) = 1 + 60 = 61 muestras
#  Con 256 muestras por latido, cubre todo el complejo QRS.
# ─────────────────────────────────────────────────────────
class TCNStack(nn.Module):
    def __init__(self, in_ch, hidden_ch=64, kernel_size=3,
                 dilations=(1, 2, 4, 8), dropout=0.1):
        super().__init__()
        layers = []
        ch = in_ch
        for d in dilations:
            layers.append(CausalDilatedBlock(ch, hidden_ch, kernel_size, d, dropout))
            ch = hidden_ch
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)    # shape preservada: (B, hidden_ch, L)


print('✅ CausalDilatedBlock y TCNStack definidos')
# Verificación rápida de shapes
_x = torch.randn(4, 6, 256)    # (Batch=4, Canales=6, L=256)
_out = TCNStack(in_ch=6, hidden_ch=64)(_x)
print(f'   Input shape : {_x.shape}')
print(f'   Output shape: {_out.shape}  (debe ser [4, 64, 256])')

## 5. Módulos TCNEncoder, Reparametrize y TCNDecoder

In [ ]:
# ─────────────────────────────────────────────────────────
#  TCNEncoder
#  Toma la señal x (B,1,256) + condición c (B,5) y produce
#  los parámetros del espacio latente: mu y log_var.
#
#  Pasos:
#    1. Expandir c a (B,5,256) y concatenar con x → (B,6,256)
#    2. Pasar por TCNStack → (B,64,256)
#    3. GlobalAveragePooling → (B,64)
#    4. Dos capas Linear independientes → mu (B,Z) y log_var (B,Z)
# ─────────────────────────────────────────────────────────
class TCNEncoder(nn.Module):
    def __init__(self, signal_len=256, n_classes=5, hidden_ch=64,
                 latent_dim=32, dropout=0.1):
        super().__init__()
        self.signal_len = signal_len
        # in_ch = 1 (señal) + 5 (one-hot expandido) = 6
        self.tcn = TCNStack(in_ch=1 + n_classes, hidden_ch=hidden_ch,
                            dropout=dropout)
        # GlobalAvgPool y proyección a espacio latente
        self.fc_mu      = nn.Linear(hidden_ch, latent_dim)
        self.fc_log_var = nn.Linear(hidden_ch, latent_dim)

    def forward(self, x, c):
        # x: (B,1,256)  c: (B,5)
        # Expandir c para concatenar en la dimensión temporal
        c_exp = c.unsqueeze(-1).expand(-1, -1, self.signal_len)  # (B,5,256)
        xc    = torch.cat([x, c_exp], dim=1)                     # (B,6,256)

        h   = self.tcn(xc)                   # (B,64,256)
        h   = h.mean(dim=-1)                 # GlobalAvgPool → (B,64)

        mu      = self.fc_mu(h)              # (B, latent_dim)
        log_var = self.fc_log_var(h)         # (B, latent_dim)
        return mu, log_var


print('✅ TCNEncoder definido')
_enc = TCNEncoder()
_x, _c = torch.randn(4,1,256), torch.randn(4,5)
_mu, _lv = _enc(_x, _c)
print(f'   mu shape     : {_mu.shape}   (B=4, latent_dim=32)')
print(f'   log_var shape: {_lv.shape}')

In [ ]:
# ─────────────────────────────────────────────────────────
#  Reparametrize
#  Truco de la reparametrización: z = mu + sigma * epsilon
#  donde epsilon ~ N(0, I) es ruido muestreado.
#
#  ¿Por qué? Si z = mu + sigma*eps, el gradiente puede fluir
#  a través de mu y sigma (deterministas), mientras que la
#  aleatoriedad queda en eps (no diferenciable, pero no necesita
#  gradiente). Sin este truco, el backpropagation no funcionaría
#  a través de una operación de muestreo.
# ─────────────────────────────────────────────────────────
class Reparametrize(nn.Module):
    def forward(self, mu, log_var):
        if self.training:
            std = torch.exp(0.5 * log_var)           # sigma = exp(0.5 * log_var)
            eps = torch.randn_like(std)               # eps ~ N(0, I)
            return mu + std * eps
        else:
            # En evaluación usamos el valor esperado (determinista)
            return mu


print('✅ Reparametrize definido')
_reparam = Reparametrize()
_reparam.train()
_z = _reparam(_mu, _lv)
print(f'   z shape (train): {_z.shape}   ← muestreado con ruido')
_reparam.eval()
_z_eval = _reparam(_mu, _lv)
print(f'   z shape (eval) : {_z_eval.shape}  ← solo mu (determinista)')

In [ ]:
# ─────────────────────────────────────────────────────────
#  TCNDecoder
#  Genera la señal sintética a partir de z + c.
#
#  Pasos:
#    1. Concat z (B,32) y c (B,5) → (B,37)
#    2. Linear → (B, 64*32) → Reshape → (B, 64, 32)
#    3. Upsampling ×3 con F.interpolate + CausalDilatedBlock:
#       (B,64,32) → (B,64,64) → (B,64,128) → (B,64,256)
#    4. Conv1d(64→1) + Sigmoid → señal en [0,1]
# ─────────────────────────────────────────────────────────
class TCNDecoder(nn.Module):
    def __init__(self, signal_len=256, n_classes=5, hidden_ch=64,
                 latent_dim=32, dropout=0.1):
        super().__init__()
        self.signal_len = signal_len
        self.hidden_ch  = hidden_ch
        self.init_len   = signal_len // 8   # 256//8 = 32

        # 1. Proyección z+c → secuencia inicial
        self.fc_in = nn.Sequential(
            nn.Linear(latent_dim + n_classes, hidden_ch * self.init_len),
            nn.GELU()
        )

        # 2. Tres bloques de upsampling: cada uno dobla la longitud
        #    ×2: 32→64  ×2: 64→128  ×2: 128→256
        self.up1 = CausalDilatedBlock(hidden_ch, hidden_ch, dilation=1, dropout=dropout)
        self.up2 = CausalDilatedBlock(hidden_ch, hidden_ch, dilation=2, dropout=dropout)
        self.up3 = CausalDilatedBlock(hidden_ch, hidden_ch, dilation=4, dropout=dropout)

        # 3. Capa de salida: proyectar a 1 canal
        self.out_conv = nn.Sequential(
            nn.Conv1d(hidden_ch, 1, kernel_size=1),
            nn.Sigmoid()   # señal normalizada [0, 1]
        )

    def forward(self, z, c):
        # z: (B, latent_dim)   c: (B, 5)
        zc = torch.cat([z, c], dim=-1)              # (B, latent_dim+5)
        h  = self.fc_in(zc)                         # (B, 64*32)
        h  = h.view(h.size(0), self.hidden_ch,
                    self.init_len)                   # (B, 64, 32)

        # Upsampling progresivo: interpolate lineal + refinamiento TCN
        h = F.interpolate(h, scale_factor=2, mode='linear', align_corners=False)  # (B,64,64)
        h = self.up1(h)

        h = F.interpolate(h, scale_factor=2, mode='linear', align_corners=False)  # (B,64,128)
        h = self.up2(h)

        h = F.interpolate(h, scale_factor=2, mode='linear', align_corners=False)  # (B,64,256)
        h = self.up3(h)

        x_hat = self.out_conv(h)                    # (B, 1, 256)
        return x_hat


print('✅ TCNDecoder definido')
_dec = TCNDecoder()
_z_test = torch.randn(4, 32)
_c_test = torch.zeros(4, 5); _c_test[:, 2] = 1.0   # clase V
_out = _dec(_z_test, _c_test)
print(f'   Input z: {_z_test.shape}  c: {_c_test.shape}')
print(f'   Output : {_out.shape}   (debe ser [4, 1, 256])')
print(f'   Rango salida: [{_out.min().item():.3f}, {_out.max().item():.3f}]  (debe estar en [0,1])')

## 6. Modelo Completo TCNCVAE + Resumen de Parámetros

In [ ]:
# ─────────────────────────────────────────────────────────
#  TCNCVAE – Modelo raíz
#  Ensambla Encoder + Reparametrize + Decoder.
#
#  forward() → (x_hat, mu, log_var)   para el bucle de entrenamiento
#  generate() → (B, 1, 256)            para inferencia sin encoder
# ─────────────────────────────────────────────────────────
class TCNCVAE(nn.Module):
    def __init__(self, signal_len=256, n_classes=5,
                 hidden_ch=64, latent_dim=32, dropout=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder    = TCNEncoder(signal_len, n_classes, hidden_ch, latent_dim, dropout)
        self.reparam    = Reparametrize()
        self.decoder    = TCNDecoder(signal_len, n_classes, hidden_ch, latent_dim, dropout)

    def forward(self, x, c):
        """Paso completo (usado durante entrenamiento)."""
        mu, log_var = self.encoder(x, c)
        z           = self.reparam(mu, log_var)
        x_hat       = self.decoder(z, c)
        return x_hat, mu, log_var

    @torch.no_grad()
    def generate(self, c, n_samples=1, device=None):
        """
        Generación pura: samplea z ~ N(0,I) y decodifica.
        c: tensor (n_samples, 5) one-hot  O  índice int escalar.
        """
        self.eval()
        dev = device or next(self.parameters()).device

        # Si se pasa un entero, construir el one-hot
        if isinstance(c, int):
            c_oh = F.one_hot(torch.tensor([c] * n_samples), num_classes=5).float().to(dev)
        else:
            c_oh = c.to(dev)

        z     = torch.randn(n_samples, self.latent_dim).to(dev)
        x_gen = self.decoder(z, c_oh)    # (n_samples, 1, 256)
        return x_gen


# ── Instanciación y resumen ───────────────────────────────
model = TCNCVAE(
    signal_len  = 256,
    n_classes   = N_CLASSES,
    hidden_ch   = 64,
    latent_dim  = 32,
    dropout     = 0.1
).to(DEVICE)

print('✅ Modelo TCNCVAE creado y enviado a', DEVICE)

# Conteo de parámetros
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n   Parámetros totales      : {total_params:,}')
print(f'   Parámetros entrenables  : {trainable_params:,}')
print(f'   Memoria aprox. (float32): {total_params * 4 / 1e6:.1f} MB')

# Resumen detallado con torchinfo
summary(model,
        input_data=[torch.randn(BATCH_SIZE, 1, 256).to(DEVICE),
                    torch.randn(BATCH_SIZE, 5).to(DEVICE)],
        col_names=['input_size', 'output_size', 'num_params'],
        depth=3, verbose=1)

## 7. Función de Pérdida y β-Scheduler

In [ ]:
# ─────────────────────────────────────────────────────────
#  FUNCIÓN DE PÉRDIDA DEL CVAE
#
#  Loss = Reconstrucción + β · KL
#
#  Reconstrucción (MSE):
#    Mide qué tan fielmente el decoder reprodujo la señal original.
#    Usamos reducción 'mean' para que la escala sea independiente
#    del batch_size y la longitud de señal.
#
#  KL Divergence:
#    Regulariza el espacio latente para que z siga una N(0,I).
#    Fórmula analítica: -0.5 * sum(1 + log_var - mu² - exp(log_var))
#    con media por latido para escala consistente.
#
#  β-Scheduler (KL Annealing):
#    Empieza en β=0 para que el encoder aprenda a reconstruir
#    sin restricción del espacio latente (evita el posterior collapse).
#    Luego sube linealmente hasta β_max.
# ─────────────────────────────────────────────────────────
def cvae_loss(x_hat, x, mu, log_var, beta=1.0):
    """
    Calcula la pérdida del CVAE.
    Retorna: loss_total, recon_loss, kl_loss (todos escalares)
    """
    # Pérdida de reconstrucción
    recon = F.mse_loss(x_hat, x, reduction='mean')

    # Divergencia KL (forma analítica)
    kl = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())

    # Pérdida combinada
    total = recon + beta * kl
    return total, recon, kl


class BetaScheduler:
    """
    Scheduler lineal para el peso β de la KL divergence.
    Durante las primeras `warmup_epochs` épocas, β=0.
    Luego sube linealmente hasta β_max en `anneal_epochs` épocas.
    """
    def __init__(self, beta_max=1.0, warmup_epochs=5, anneal_epochs=20):
        self.beta_max      = beta_max
        self.warmup_epochs = warmup_epochs
        self.anneal_epochs = anneal_epochs

    def get_beta(self, epoch):
        if epoch < self.warmup_epochs:
            return 0.0
        elif epoch < self.warmup_epochs + self.anneal_epochs:
            progress = (epoch - self.warmup_epochs) / self.anneal_epochs
            return self.beta_max * progress
        else:
            return self.beta_max


# ── Visualización del β-schedule ─────────────────────────
scheduler_preview = BetaScheduler(beta_max=1.0, warmup_epochs=5, anneal_epochs=20)
epochs_preview = range(50)
betas_preview  = [scheduler_preview.get_beta(e) for e in epochs_preview]

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(epochs_preview, betas_preview, color='#9b59b6', linewidth=2)
ax.axvspan(0,  5, alpha=0.08, color='#e74c3c', label='Warmup (β=0): solo reconstrucción')
ax.axvspan(5, 25, alpha=0.08, color='#f39c12', label='Annealing lineal (β: 0→1)')
ax.axvspan(25,50, alpha=0.08, color='#27ae60', label='Entrenamiento estable (β=1)')
ax.set_xlabel('Época')
ax.set_ylabel('β (peso KL)')
ax.set_title('β-Scheduler: KL Annealing', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('beta_schedule.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Función de pérdida y β-Scheduler definidos')

## 8. Bucle de Entrenamiento

In [ ]:
# ─────────────────────────────────────────────────────────
#  HIPERPARÁMETROS DE ENTRENAMIENTO
# ─────────────────────────────────────────────────────────
N_EPOCHS      = 50       # Ajustar según tiempo disponible en Colab
LR            = 1e-3     # Learning rate inicial para AdamW
WEIGHT_DECAY  = 1e-4     # Regularización L2
GRAD_CLIP     = 1.0      # Gradient clipping (previene explosión en TCN profundo)
BETA_MAX      = 1.0
WARMUP_EPOCHS = 5
ANNEAL_EPOCHS = 20

# Directorio para checkpoints
CKPT_DIR = '/content/tcncvae_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Optimizador y LR Scheduler ───────────────────────────
optimizer = torch.optim.AdamW(model.parameters(),
                               lr=LR, weight_decay=WEIGHT_DECAY)

# CosineAnnealingLR: decae el LR suavemente, útil para VAEs
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-5
)

beta_scheduler = BetaScheduler(BETA_MAX, WARMUP_EPOCHS, ANNEAL_EPOCHS)

print('✅ Configuración de entrenamiento:')
print(f'   Épocas        : {N_EPOCHS}')
print(f'   Batch size    : {BATCH_SIZE}')
print(f'   Optimizador   : AdamW  lr={LR}  wd={WEIGHT_DECAY}')
print(f'   LR Scheduler  : CosineAnnealingLR  T_max={N_EPOCHS}')
print(f'   Grad clip     : {GRAD_CLIP}')
print(f'   β-schedule    : warmup={WARMUP_EPOCHS}  anneal={ANNEAL_EPOCHS}  β_max={BETA_MAX}')
print(f'   Checkpoints   : {CKPT_DIR}')

In [ ]:
# ─────────────────────────────────────────────────────────
#  FUNCIONES DE TRAIN / VALIDATE POR ÉPOCA
# ─────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, beta, device):
    model.train()
    total_loss = recon_loss_sum = kl_loss_sum = 0.0

    for x, c, _ in loader:
        x, c = x.to(device), c.to(device)

        optimizer.zero_grad()
        x_hat, mu, log_var = model(x, c)
        loss, recon, kl    = cvae_loss(x_hat, x, mu, log_var, beta)

        loss.backward()
        # Gradient clipping: evita explosión en TCN profundo
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        total_loss    += loss.item()
        recon_loss_sum += recon.item()
        kl_loss_sum    += kl.item()

    n = len(loader)
    return total_loss/n, recon_loss_sum/n, kl_loss_sum/n


@torch.no_grad()
def validate(model, loader, beta, device):
    model.eval()
    total_loss = recon_loss_sum = kl_loss_sum = 0.0

    for x, c, _ in loader:
        x, c = x.to(device), c.to(device)
        x_hat, mu, log_var = model(x, c)
        loss, recon, kl    = cvae_loss(x_hat, x, mu, log_var, beta)
        total_loss     += loss.item()
        recon_loss_sum += recon.item()
        kl_loss_sum    += kl.item()

    n = len(loader)
    return total_loss/n, recon_loss_sum/n, kl_loss_sum/n


# ─────────────────────────────────────────────────────────
#  BUCLE PRINCIPAL DE ENTRENAMIENTO
# ─────────────────────────────────────────────────────────
history = {
    'train_loss': [], 'val_loss': [],
    'train_recon': [], 'val_recon': [],
    'train_kl': [],   'val_kl': [],
    'beta': [], 'lr': []
}

best_val_loss  = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())

print('🚀 Iniciando entrenamiento...\n')
print(f'{"Epoch":>6} {"β":>5} {"Train Loss":>12} {"Recon":>9} {"KL":>7}  '
      f'{"Val Loss":>10} {"Recon":>9} {"KL":>7}  {"LR":>8}  {"t(s)":>5}')
print('─' * 100)

for epoch in range(N_EPOCHS):
    t0   = time.time()
    beta = beta_scheduler.get_beta(epoch)
    current_lr = optimizer.param_groups[0]['lr']

    tr_loss, tr_recon, tr_kl = train_one_epoch(model, train_loader, optimizer, beta, DEVICE)
    vl_loss, vl_recon, vl_kl = validate(model, val_loader, beta, DEVICE)

    lr_scheduler.step()
    elapsed = time.time() - t0

    # Guardar historial
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_recon'].append(tr_recon)
    history['val_recon'].append(vl_recon)
    history['train_kl'].append(tr_kl)
    history['val_kl'].append(vl_kl)
    history['beta'].append(beta)
    history['lr'].append(current_lr)

    # Guardar mejor modelo
    if vl_loss < best_val_loss:
        best_val_loss  = vl_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save({'epoch': epoch,
                    'model_state_dict': best_model_wts,
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_loss': best_val_loss,
                    'history': history},
                   f'{CKPT_DIR}/best_model.pt')
        flag = ' ✅ mejor'
    else:
        flag = ''

    print(f'{epoch+1:>6} {beta:>5.2f} {tr_loss:>12.5f} {tr_recon:>9.5f} {tr_kl:>7.4f}  '
          f'{vl_loss:>10.5f} {vl_recon:>9.5f} {vl_kl:>7.4f}  '
          f'{current_lr:>8.2e}  {elapsed:>4.1f}s{flag}')

# Restaurar mejor modelo
model.load_state_dict(best_model_wts)
print(f'\n🏆 Entrenamiento completado. Mejor val_loss = {best_val_loss:.5f}')
print(f'   Checkpoint guardado en {CKPT_DIR}/best_model.pt')

In [ ]:
# ─────────────────────────────────────────────────────────
#  CURVAS DE PÉRDIDA
# ─────────────────────────────────────────────────────────
epochs_range = range(1, N_EPOCHS + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Curvas de Entrenamiento – TCN-CVAE', fontsize=13, fontweight='bold')

# Loss total
ax = axes[0, 0]
ax.plot(epochs_range, history['train_loss'], label='Train', color='#e74c3c', linewidth=1.8)
ax.plot(epochs_range, history['val_loss'],   label='Val',   color='#3498db', linewidth=1.8)
ax.set_title('Loss Total (MSE + β·KL)')
ax.set_xlabel('Época'); ax.set_ylabel('Loss'); ax.legend()

# Reconstrucción
ax = axes[0, 1]
ax.plot(epochs_range, history['train_recon'], label='Train', color='#e74c3c', linewidth=1.8)
ax.plot(epochs_range, history['val_recon'],   label='Val',   color='#3498db', linewidth=1.8)
ax.set_title('Loss de Reconstrucción (MSE)')
ax.set_xlabel('Época'); ax.set_ylabel('MSE'); ax.legend()

# KL divergence
ax = axes[1, 0]
ax.plot(epochs_range, history['train_kl'], label='Train', color='#9b59b6', linewidth=1.8)
ax.plot(epochs_range, history['val_kl'],   label='Val',   color='#1abc9c', linewidth=1.8)
ax.set_title('KL Divergence')
ax.set_xlabel('Época'); ax.set_ylabel('KL'); ax.legend()

# β y LR
ax  = axes[1, 1]
ax2 = ax.twinx()
ax.plot(epochs_range, history['beta'], color='#e67e22', linewidth=2, label='β')
ax2.plot(epochs_range, history['lr'], color='#2c3e50', linewidth=1.5,
         linestyle='--', label='LR')
ax.set_title('β-schedule y Learning Rate')
ax.set_xlabel('Época')
ax.set_ylabel('β', color='#e67e22')
ax2.set_ylabel('Learning Rate', color='#2c3e50')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=9)

plt.tight_layout()
plt.savefig('curvas_entrenamiento.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Generación de Latidos Sintéticos Condicionales

In [ ]:
# ─────────────────────────────────────────────────────────
#  GENERACIÓN SINTÉTICA POR CLASE AAMI
#  Para cada una de las 5 clases, se genera a partir de z ~ N(0,I)
#  y el vector condicional c correspondiente.
#  El encoder NO interviene en inferencia.
# ─────────────────────────────────────────────────────────
N_GEN = 6  # Latidos sintéticos por clase

class_colors = {
    'N': '#27ae60', 'S': '#2980b9', 'V': '#e74c3c',
    'F': '#e67e22', 'Q': '#7f8c8d'
}
class_desc = {
    'N': 'Normal', 'S': 'Supraventricular',
    'V': 'Ventricular', 'F': 'Fusión', 'Q': 'Desconocido'
}

model.eval()
t_ms = np.arange(256) / 360 * 1000  # eje temporal en ms

fig, axes = plt.subplots(N_CLASSES, N_GEN, figsize=(15, 2.4 * N_CLASSES))
fig.suptitle('Latidos Sintéticos Generados por Clase AAMI (TCN-CVAE)',
             fontsize=12, fontweight='bold')

for row, (cls, cls_idx) in enumerate(CLASS_IDX.items()):
    # Generar N_GEN latidos sintéticos de esta clase
    x_syn = model.generate(c=cls_idx, n_samples=N_GEN, device=DEVICE)
    x_syn = x_syn.squeeze(1).cpu().numpy()   # (N_GEN, 256)

    for col in range(N_GEN):
        ax = axes[row, col]
        ax.plot(t_ms, x_syn[col], color=class_colors[cls], linewidth=1.2)
        ax.axvline(x=256/2/360*1000, color='gray', linestyle='--',
                   alpha=0.4, linewidth=0.8)
        ax.set_ylim([-0.05, 1.05])
        ax.tick_params(labelsize=7)
        if col == 0:
            ax.set_ylabel(f'{cls}\n{class_desc[cls]}',
                          fontsize=9, fontweight='bold')
        if row == 0:
            ax.set_title(f'Sintético {col+1}', fontsize=9)
        if row == N_CLASSES - 1:
            ax.set_xlabel('ms')

plt.tight_layout()
plt.savefig('latidos_sinteticos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Latidos sintéticos generados por clase AAMI.')

In [ ]:
# ─────────────────────────────────────────────────────────
#  COMPARACIÓN: REAL vs SINTÉTICO
#  Para las 3 clases más importantes (N, V, S),
#  se comparan 3 latidos reales contra 3 sintéticos.
# ─────────────────────────────────────────────────────────
COMPARE_CLASSES = ['N', 'V', 'S']
N_COMP = 3

fig, axes = plt.subplots(len(COMPARE_CLASSES), N_COMP * 2,
                          figsize=(15, 2.8 * len(COMPARE_CLASSES)))
fig.suptitle('Comparación: Latidos Reales (izq.) vs Sintéticos (der.) por clase',
             fontsize=12, fontweight='bold')

for row, cls in enumerate(COMPARE_CLASSES):
    cls_idx = CLASS_IDX[cls]
    col_color = class_colors[cls]

    # Latidos reales del set de test
    real_idx = np.where(y_test == cls)[0]
    chosen   = np.random.choice(real_idx, size=N_COMP, replace=False)
    X_real   = X_test[chosen]     # (N_COMP, 256)

    # Latidos sintéticos
    x_syn = model.generate(c=cls_idx, n_samples=N_COMP, device=DEVICE)
    X_syn = x_syn.squeeze(1).cpu().numpy()   # (N_COMP, 256)

    for col in range(N_COMP):
        # Real
        ax = axes[row, col]
        ax.plot(t_ms, X_real[col], color=col_color, linewidth=1.2)
        ax.set_ylim([-0.05, 1.05])
        ax.tick_params(labelsize=7)
        if col == 0:
            ax.set_ylabel(f'{cls} – {class_desc[cls]}\n(REAL)',
                          fontsize=8, fontweight='bold')
        if row == 0: ax.set_title(f'Real {col+1}', fontsize=9, color='#444')
        if row == len(COMPARE_CLASSES)-1: ax.set_xlabel('ms')

        # Sintético
        ax = axes[row, col + N_COMP]
        ax.plot(t_ms, X_syn[col], color=col_color, linewidth=1.2,
                linestyle='--', alpha=0.85)
        ax.set_ylim([-0.05, 1.05])
        ax.tick_params(labelsize=7)
        if col == 0:
            ax.set_ylabel(f'{cls} – {class_desc[cls]}\n(SINTÉTICO)',
                          fontsize=8, fontweight='bold', color='#555')
        if row == 0: ax.set_title(f'Sintético {col+1}', fontsize=9, color='#888')
        if row == len(COMPARE_CLASSES)-1: ax.set_xlabel('ms')

plt.tight_layout()
plt.savefig('comparacion_real_sintetico.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Validación de Calidad Generativa

In [ ]:
# ─────────────────────────────────────────────────────────
#  ANÁLISIS ESPECTRAL: REAL vs SINTÉTICO
#  Compara la densidad espectral de potencia (PSD) media
#  para las 5 clases. Si el modelo aprendió bien la morfología,
#  los espectros reales y sintéticos deben solaparse.
# ─────────────────────────────────────────────────────────
from scipy.signal import welch

FS = 360
N_SPECTRAL = 200  # Latidos por clase para estimar el PSD medio

fig, axes = plt.subplots(1, N_CLASSES, figsize=(16, 4), sharey=True)
fig.suptitle('Análisis Espectral PSD: Reales vs Sintéticos por clase AAMI',
             fontsize=11, fontweight='bold')

model.eval()
for col, cls in enumerate(CLASS_NAMES):
    cls_idx = CLASS_IDX[cls]
    ax = axes[col]

    # PSD de latidos reales del test set
    real_idx  = np.where(y_test == cls)[0]
    n_samples = min(N_SPECTRAL, len(real_idx))
    chosen    = np.random.choice(real_idx, size=n_samples, replace=False)
    psds_real = []
    for i in chosen:
        f, p = welch(X_test[i], fs=FS, nperseg=64)
        psds_real.append(p)
    psd_real_mean = np.mean(psds_real, axis=0)
    psd_real_std  = np.std(psds_real,  axis=0)

    # PSD de latidos sintéticos
    x_syn = model.generate(c=cls_idx, n_samples=n_samples, device=DEVICE)
    X_syn = x_syn.squeeze(1).cpu().numpy()
    psds_syn = []
    for s in X_syn:
        f, p = welch(s, fs=FS, nperseg=64)
        psds_syn.append(p)
    psd_syn_mean = np.mean(psds_syn, axis=0)
    psd_syn_std  = np.std(psds_syn,  axis=0)

    # Plot
    ax.semilogy(f, psd_real_mean, color=class_colors[cls],
                linewidth=2, label='Real')
    ax.fill_between(f, psd_real_mean-psd_real_std,
                    psd_real_mean+psd_real_std,
                    alpha=0.2, color=class_colors[cls])
    ax.semilogy(f, psd_syn_mean, color=class_colors[cls],
                linewidth=1.5, linestyle='--', alpha=0.7, label='Sintético')
    ax.fill_between(f, psd_syn_mean-psd_syn_std,
                    psd_syn_mean+psd_syn_std,
                    alpha=0.1, color=class_colors[cls], hatch='//')

    ax.set_title(f'Clase {cls}\n{class_desc[cls]}', fontsize=9)
    ax.set_xlabel('Hz')
    if col == 0: ax.set_ylabel('PSD')
    ax.legend(fontsize=8)
    ax.set_xlim([0, FS/2])

plt.tight_layout()
plt.savefig('analisis_espectral.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Análisis espectral completado. Los espectros solapados indican buena calidad generativa.')

In [ ]:
# ─────────────────────────────────────────────────────────
#  t-SNE DEL ESPACIO LATENTE
#  Visualiza cómo se organizan las clases en el espacio z.
#  Si el CVAE aprendió bien:
#    - Los latidos reales de cada clase formarán clusters separados.
#    - Los z sintéticos sampleados de N(0,I) se mezclarán
#      con los reales de su misma clase.
# ─────────────────────────────────────────────────────────
N_TSNE = 300  # Latidos por clase (para que t-SNE sea manejable)

zs_real  = []
zs_syn   = []
labs_real = []
labs_syn  = []

model.eval()
with torch.no_grad():
    for cls in CLASS_NAMES:
        cls_idx = CLASS_IDX[cls]

        # Latidos reales → pasar por encoder → obtener mu como representación z
        real_idx = np.where(y_test == cls)[0]
        n = min(N_TSNE, len(real_idx))
        chosen = np.random.choice(real_idx, size=n, replace=False)

        x_r  = torch.tensor(X_test[chosen], dtype=torch.float32).unsqueeze(1).to(DEVICE)
        c_r  = F.one_hot(torch.tensor([cls_idx]*n), num_classes=5).float().to(DEVICE)
        mu_r, _ = model.encoder(x_r, c_r)
        zs_real.append(mu_r.cpu().numpy())
        labs_real.extend([cls] * n)

        # Latidos sintéticos → samplear z ~ N(0,I) directamente
        z_s = torch.randn(n, model.latent_dim).to(DEVICE)
        zs_syn.append(z_s.cpu().numpy())
        labs_syn.extend([cls] * n)

Z_real = np.concatenate(zs_real, axis=0)  # (N_total, latent_dim)
Z_syn  = np.concatenate(zs_syn,  axis=0)
Z_all  = np.concatenate([Z_real, Z_syn], axis=0)
origin = ['Real'] * len(Z_real) + ['Sintético'] * len(Z_syn)
labels_all = labs_real + labs_syn

# Reducción t-SNE a 2D
print('⏳ Calculando t-SNE (puede tardar ~30s)...')
tsne  = TSNE(n_components=2, perplexity=40, n_iter=800,
             random_state=SEED, verbose=0)
Z_2d  = tsne.fit_transform(Z_all)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('t-SNE del Espacio Latente z – TCN-CVAE', fontsize=12, fontweight='bold')

# Por clase AAMI
for cls in CLASS_NAMES:
    mask = [l == cls for l in labels_all]
    ax1.scatter(Z_2d[mask, 0], Z_2d[mask, 1],
               c=class_colors[cls], s=8, alpha=0.5, label=f'{cls} ({class_desc[cls]})')
ax1.set_title('Coloreado por clase AAMI')
ax1.legend(fontsize=8, markerscale=2)
ax1.set_xlabel('t-SNE dim 1'); ax1.set_ylabel('t-SNE dim 2')

# Real vs Sintético
for orig, col, mk in [('Real', '#2c3e50', 'o'), ('Sintético', '#e74c3c', '^')]:
    mask = [o == orig for o in origin]
    ax2.scatter(Z_2d[mask, 0], Z_2d[mask, 1],
               c=col, s=8, alpha=0.4, marker=mk, label=orig)
ax2.set_title('Coloreado por origen (Real vs Sintético)')
ax2.legend(fontsize=9, markerscale=2)
ax2.set_xlabel('t-SNE dim 1'); ax2.set_ylabel('t-SNE dim 2')

plt.tight_layout()
plt.savefig('tsne_latente.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ t-SNE completado.')
print('   Si los clusters de clases están separados → el CVAE aprendió representaciones discriminativas.')
print('   Si Real y Sintético se mezclan → el decoder genera distribuciones realistas.')

In [ ]:
# ─────────────────────────────────────────────────────────
#  EXPORTAR LATIDOS SINTÉTICOS PARA USO EN AUGMENTATION
#  Genera N latidos por clase y los guarda en .npy
# ─────────────────────────────────────────────────────────
N_EXPORT_PER_CLASS = 1000  # Latidos sintéticos por clase a exportar

syn_dir = '/content/ecg_synthetic'
os.makedirs(syn_dir, exist_ok=True)

X_synthetic = []
y_synthetic = []

model.eval()
print(f'⏳ Generando {N_EXPORT_PER_CLASS} latidos sintéticos por clase...')
for cls in CLASS_NAMES:
    cls_idx = CLASS_IDX[cls]
    # Generar en batches de 256 para no saturar la VRAM
    batches  = []
    remaining = N_EXPORT_PER_CLASS
    while remaining > 0:
        n_batch = min(256, remaining)
        x_b = model.generate(c=cls_idx, n_samples=n_batch, device=DEVICE)
        batches.append(x_b.squeeze(1).cpu().numpy())
        remaining -= n_batch
    X_cls = np.concatenate(batches, axis=0)  # (N_EXPORT, 256)
    X_synthetic.append(X_cls)
    y_synthetic.extend([cls] * N_EXPORT_PER_CLASS)
    print(f'   Clase {cls} ({class_desc[cls]}): {len(X_cls)} latidos generados')

X_synthetic = np.concatenate(X_synthetic, axis=0)  # (5*N_EXPORT, 256)
y_synthetic = np.array(y_synthetic)

np.save(f'{syn_dir}/X_synthetic.npy', X_synthetic)
np.save(f'{syn_dir}/y_synthetic.npy', y_synthetic)

print(f'\n✅ Dataset sintético guardado en {syn_dir}/')
print(f'   X_synthetic shape: {X_synthetic.shape}')
print(f'   y_synthetic shape: {y_synthetic.shape}')
print(f'   Distribución: {dict(Counter(y_synthetic))}')
print(f'   Memoria: {X_synthetic.nbytes / 1e6:.1f} MB')

In [ ]:
# ─────────────────────────────────────────────────────────
#  RESUMEN FINAL DEL ENTRENAMIENTO
# ─────────────────────────────────────────────────────────
print('=' * 65)
print('          RESUMEN – TCN-CVAE ENTRENAMIENTO COMPLETO')
print('=' * 65)

print('\n🏗️  ARQUITECTURA')
print(f'   Señal de entrada  : (1, 256) – ECG normalizado [0,1]')
print(f'   Condición c       : one-hot (5 clases AAMI)')
print(f'   Dimensión latente : {model.latent_dim}')
print(f'   Canales ocultos   : 64')
print(f'   Dilataciones TCN  : [1, 2, 4, 8]')
print(f'   Parámetros        : {sum(p.numel() for p in model.parameters()):,}')

print('\n📉  ENTRENAMIENTO')
print(f'   Épocas            : {N_EPOCHS}')
print(f'   Mejor val_loss    : {best_val_loss:.5f}')
best_epoch = history["val_loss"].index(min(history["val_loss"])) + 1
print(f'   Mejor época       : {best_epoch}')
print(f'   β final           : {history["beta"][-1]:.2f}')
final_lr = history['lr'][-1]
print(f'   LR final          : {final_lr:.2e}')

print('\n🎲  GENERACIÓN SINTÉTICA')
print(f'   Latidos exportados: {len(X_synthetic):,}')
print(f'   Por clase         : {N_EXPORT_PER_CLASS} por clase × 5 clases')
print(f'   Guardado en       : /content/ecg_synthetic/')

print('\n📁  ARCHIVOS GENERADOS')
for fname in ['beta_schedule.png', 'curvas_entrenamiento.png',
              'latidos_sinteticos.png', 'comparacion_real_sintetico.png',
              'analisis_espectral.png', 'tsne_latente.png']:
    exists = '✅' if os.path.exists(fname) else '❌'
    print(f'   {exists}  {fname}')

print('=' * 65)

---
## ✅ Fin del Notebook de Entrenamiento

**Próximos pasos sugeridos:**
- Combinar `X_train_bal` con `X_synthetic` y evaluar si el clasificador downstream mejora.
- Experimentar con `latent_dim=64` o más bloques TCN si el MSE de reconstrucción no converge.
- Ajustar `beta_max` a valores más pequeños (0.1–0.5) si las señales generadas pierden detalle morfológico.